# Jira Delivery Data Validation

In [1]:
import pandas as pd

In [2]:
## set data path
data_path = '../../data/raw/'

## load each csv into a dataframe
projects = pd.read_csv(data_path + 'projects.csv')
teams = pd.read_csv(data_path + 'teams.csv')
team_members = pd.read_csv(data_path + 'team_members.csv')
epics = pd.read_csv(data_path + 'epics.csv')
issues = pd.read_csv(data_path + 'issues.csv')
issue_status_history = pd.read_csv(data_path + 'issue_status_history.csv')
sprints = pd.read_csv(data_path + 'sprints.csv')
releases = pd.read_csv(data_path + 'releases.csv')
release_issues = pd.read_csv(data_path + 'release_issues.csv')



In [3]:
## create dataframe of all dataframes - used to check summary status for all dataframes
tables = {
    'projects': projects,
    'teams': teams,
    'team_members': team_members,
    'epics': epics,
    'issues': issues,
    'issue_status_history': issue_status_history,
    'sprints': sprints,
    'releases': releases,
    'release_issues': release_issues
}

In [4]:
## display shape of all imported csv files
for name, df in tables.items():
    print(name, df.shape)

projects (2, 4)
teams (4, 4)
team_members (18, 4)
epics (50, 5)
issues (500, 14)
issue_status_history (3152, 3)
sprints (104, 5)
releases (8, 4)
release_issues (244, 2)


In [5]:
## display column names for each table 
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


PROJECTS
['project_id', 'project_key', 'project_name', 'owner_team_id']

TEAMS
['team_id', 'team_name', 'specialization', 'team_type']

TEAM_MEMBERS
['member_id', 'team_id', 'role', 'seniority']

EPICS
['epic_id', 'project_id', 'team_id', 'epic_key', 'epic_name']

ISSUES
['issue_id', 'issue_key', 'project_id', 'team_id', 'epic_id', 'assignee_id', 'issue_type', 'priority', 'summary', 'created_at', 'resolved_at', 'story_points', 'current_status', 'sprint_id']

ISSUE_STATUS_HISTORY
['issue_id', 'status', 'changed_at']

SPRINTS
['sprint_id', 'project_id', 'sprint_name', 'start_date', 'end_date']

RELEASES
['release_id', 'project_id', 'release_name', 'release_date']

RELEASE_ISSUES
['release_id', 'issue_id']


In [6]:
## analyze release issues for grain
release_issues.columns.tolist()
release_issues.head(20).sort_values('release_id')

,release_id,issue_id
1,1,4
3,1,6
13,1,27
10,1,21
18,1,36
19,1,39
17,1,34
6,2,12
16,2,32
11,2,22


In [7]:
## display the count of each record value
print(release_issues['issue_id'].value_counts().head(10))
print(release_issues['release_id'].value_counts().head(10))

issue_id
1     1
4     1
5     1
6     1
8     1
11    1
12    1
14    1
15    1
18    1
Name: count, dtype: int64
release_id
8    39
5    37
2    36
4    31
7    30
1    25
3    23
6    23
Name: count, dtype: int64


In [8]:
## check for duplicate issue_id to determine grain
print(release_issues['issue_id'].duplicated().sum())

0


## Data Validation

Validate source data before database creation.

Validation areas:
- Missing-value validity
- Numeric, categorical, and date anomalies
- Primary and forign key integrity
- Data types and schema compatibility

In [112]:
## Loop dataframe to summarize common data anomoly column attributes
tables_validation = pd.DataFrame(columns=["TABLE",'ROWS','COLUMNS','DUPLICATES','NULL COLS','TOTAL NULLS'])

for name, df in tables.items():
    table_name = name
    row_count = df.shape[0]
    column_count = df.shape[1]
    duplicate_count = df.duplicated().sum()
    null_columns = df.isna().any().sum()
    total_nulls = df.isna().sum().sum()

    tables_validation.loc[len(tables_validation)] = [
        table_name,
        row_count,
        column_count,
        duplicate_count,
        null_columns,
        total_nulls
    ]

tables_validation


,TABLE,ROWS,COLUMNS,DUPLICATES,NULL COLS,TOTAL NULLS
0,projects,2,4,0,0,0
1,teams,4,4,0,0,0
2,team_members,18,4,0,0,0
3,epics,50,5,0,0,0
4,issues,500,14,0,1,85
5,issue_status_history,3152,3,104,0,0
6,sprints,104,5,0,0,0
7,releases,8,4,0,0,0
8,release_issues,244,2,0,0,0


In [113]:
## review Duplicated anaomolies
null_counts = issues.isna().sum()

##find columns in dataframe that contain null values
null_columns = null_counts[null_counts > 0].index

#create dataframe of all rows with Nulls
issues_with_nulls = issues[
    issues[null_columns].isna().any(axis=1)
]

issues_with_nulls


,issue_id,issue_key,project_id,team_id,epic_id,assignee_id,issue_type,priority,summary,created_at,resolved_at,story_points,current_status,sprint_id
6,7,PROJA-00007,1,1,1,3,Story,P1,Refine accessibility UX,2024-11-11 10:28:00,NaN,13,Backlog,23
9,10,PROJA-00010,1,1,1,5,Story,P3,Add audit log module,2024-06-09 16:36:00,NaN,2,In Progress,12
18,19,PROJA-00019,1,1,2,5,Story,P1,Refine profile UX,2025-09-22 16:53:00,NaN,3,Backlog,46
19,20,PROJA-00020,1,1,2,1,Story,P4,Build reports page,2025-07-06 13:59:00,NaN,8,In Progress,40
23,24,PROJA-00024,1,1,3,5,Story,P3,Implement billing workflow,2025-04-25 14:11:00,NaN,5,Selected for Development,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,473,PROJB-00473,2,4,48,14,Bug,P3,Patch Android deep link bug in export,2025-04-21 14:59:00,NaN,3,Backlog,87
476,477,PROJB-00477,2,4,48,15,Story,P3,Refactor Android search flow,2024-01-31 16:59:00,NaN,8,Blocked,55
484,485,PROJB-00485,2,4,49,16,Story,P3,Add Kotlin UI for payments,2025-01-30 09:18:00,NaN,1,Selected for Development,81
497,498,PROJB-00498,2,4,50,16,Bug,P2,Patch Android deep link bug in accessibility,2025-08-28 08:34:00,NaN,5,Selected for Development,96


In [114]:
#deteremine statuses to identify which are valid for null values
issues['current_status'].value_counts().sort_index()

current_status
Backlog                      38
Blocked                      12
Done                        415
In Progress                  13
Selected for Development     22
Name: count, dtype: int64

In [115]:
issues_with_nulls.loc[
    (issues_with_nulls['current_status'] == 'Done') &
    (issues_with_nulls['resolved_at'].isna())
]

,issue_id,issue_key,project_id,team_id,epic_id,assignee_id,issue_type,priority,summary,created_at,resolved_at,story_points,current_status,sprint_id


In [116]:
## validate Nulls count
total_issues = issues.shape[0]
done = len(issues[issues['current_status'] == 'Done'])
not_done = len(issues[issues['current_status'] != 'Done'])
resolved_at_null = issues['resolved_at'].isna().sum()

done_resolved_at_null = len(
    issues[
        (issues['current_status'] == 'Done') & 
        (issues['resolved_at'].isna())
    ]
)

not_done_resolved_null = len(
    issues[
        (issues['current_status'] != 'Done') & 
        ~(issues['resolved_at'].isna())
    ]
)

print("Total issues:             ", total_issues)
print("Done:                     ", done)
print("Not Done:                 ", not_done)
print("resolved_at NULL:         ", resolved_at_null)
print("Done + resolved_at NULL:  ", done_resolved_at_null)
print("Not Done + resolved_at populated: ", not_done_resolved_null)

Total issues:              500
Done:                      415
Not Done:                  85
resolved_at NULL:          85
Done + resolved_at NULL:   0
Not Done + resolved_at populated:  0


**Validation finding:** `resolved_at` contains 85 null values. All 85 correspond to issues not currently in `Done` status. No `Done` issues have a null `resolved_at`. Null values are therefore consistent with the issue lifecycle and require no remediation.

In [117]:
## check duplicates
duplicated_status_record = issue_status_history[
    issue_status_history.duplicated(keep=False)
]

duplicated_status_record = duplicated_status_record.sort_values(
    ['issue_id','changed_at']
)

print('Number of duplicated records: ', len(duplicated_status_record))
print('Number of extra rows due to duplication: ', issue_status_history.duplicated().sum())
print('')
print(duplicated_status_record.head(10))

Number of duplicated records:  208
Number of extra rows due to duplication:  104

    issue_id status           changed_at
5          1   Done  2025-08-19 13:23:00
6          1   Done  2025-08-19 13:23:00
26         4   Done  2025-02-06 15:46:00
27         4   Done  2025-02-06 15:46:00
40         6   Done  2024-12-15 18:40:00
41         6   Done  2024-12-15 18:40:00
67        11   Done  2025-12-08 11:14:00
68        11   Done  2025-12-08 11:14:00
88        14   Done  2025-11-22 20:53:00
89        14   Done  2025-11-22 20:53:00


Validation finding under investigation:  `issue_status_history` contains 104 additional duplicate rows, representing 208 records across duplicate groups.  Initial inspection shows identical `issue_id`, `status` and `changed_at` values.  Because this table represents status history, duplicates will not be removed until their relationship to valid issue lifecycle events is assessed.

In [118]:
## validate process flow to investigate duplication
duplicate_population = (
    duplicated_status_record
    .drop_duplicates()
    .reset_index()
)

duplicate_issue_status_history = issue_status_history.merge(duplicate_population, how='inner',on=('issue_id'),suffixes=('_orig','_dup'))

print(duplicate_issue_status_history[['issue_id','status_orig','changed_at_orig']].sort_values(['issue_id','changed_at_orig']).head(20))

    issue_id               status_orig      changed_at_orig
0          1                   Backlog  2025-08-02 11:23:00
1          1  Selected for Development  2025-08-06 13:23:00
2          1               In Progress  2025-08-10 16:23:00
3          1               Code Review  2025-08-16 20:23:00
4          1                        QA  2025-08-19 00:23:00
5          1                      Done  2025-08-19 13:23:00
6          1                      Done  2025-08-19 13:23:00
7          4                   Backlog  2025-01-28 11:46:00
8          4  Selected for Development  2025-01-29 16:46:00
9          4               In Progress  2025-01-30 18:46:00
10         4               Code Review  2025-01-31 21:46:00
11         4                        QA  2025-02-04 02:46:00
12         4                      Done  2025-02-06 15:46:00
13         4                      Done  2025-02-06 15:46:00
14         6                   Backlog  2024-11-28 17:40:00
15         6  Selected for Development  

In [119]:
## clean duplicates and provide summary analysis
issue_status_history_clean = issue_status_history.drop_duplicates().copy()

print('Raw rows:               ', len(issue_status_history))
print('Duplicates removed:     ', issue_status_history.duplicated().sum())
print('Clean rows:             ', len(issue_status_history_clean))
print('Remaining Duplicates:   ', issue_status_history_clean.duplicated().sum())
print('original vs clean Row Delta ', len(issue_status_history) - len(issue_status_history_clean))

Raw rows:                3152
Duplicates removed:      104
Clean rows:              3048
Remaining Duplicates:    0
original vs clean Row Delta  104


Cleaning decision: `issue_status_history` contained 104 exact duplicate records across `issue_id`, `status`, and `changed_at`.  Because no available field distinguishes these rows as sperate status events, retaining them would duplicate identitical lifecycle events and could distort downstream status history analysis.  Exact duplicates were removed from the processed dataset.  The original source file remains unchanged.

### Primary Key Validation

Validate that each expected primary key uniquely identifies a record and contains no null values.  Composite primary keys are validated using the complete key combination.

In [120]:
## Create list of primary keys, validate data integrity and summarize findings
primary_keys = {
    'projects': ['project_id'],
    'teams': ['team_id'],
    'team_members': ['member_id'],
    'epics': ['epic_id'],
    'issues': ['issue_id'],
    'sprints': ['sprint_id'],
    'releases': ['release_id'],
    'release_issues': ['release_id','issue_id']
}

#create empty list
results = []

# loop through expected primary keys and build validation summary
for name, pk in primary_keys.items():

    nulls = tables[name][pk].isna().sum().sum()
    dups = tables[name][pk].duplicated().sum()

    row = {
        'TABLE': name,
        'PRIMARY_KEY': ' + '.join(pk),
        'NULLS': nulls, 
        'DUPLICATES': dups,
        'STATUS': 'PASSED' if nulls == 0 and dups == 0 else 'FAILED'
    }
    results.append(row)

pk_summary_validation = pd.DataFrame(results)

print(pk_summary_validation)


            TABLE            PRIMARY_KEY  NULLS  DUPLICATES  STATUS
0        projects             project_id      0           0  PASSED
1           teams                team_id      0           0  PASSED
2    team_members              member_id      0           0  PASSED
3           epics                epic_id      0           0  PASSED
4          issues               issue_id      0           0  PASSED
5         sprints              sprint_id      0           0  PASSED
6        releases             release_id      0           0  PASSED
7  release_issues  release_id + issue_id      0           0  PASSED


Validation finding: All expected primary keys contain zero null values and zero duplicate key combinations.  The composite primary key (`release_id`, `issue_id`) in `release_issues` is also unique.  Primary key integrity checks passed for all applicable tables.

### Foreign Key Validation

Validate each expected foreign key against it's respective primary key against two key validation rules:

- data type matches between foreign key and it's respective primary key
- data reconciliation between foreign key and primary key counterpart

In [121]:
# Create foreign key structure
fk = {
    'projects': (['owner_team_id'], ['teams']),
    'team_members': (['team_id'],['teams']),
    'epics': (['project_id','team_id'],['projects','teams']),
    'issues': (['project_id','team_id','epic_id','assignee_id','sprint_id'],['projects','teams','epics','team_members','sprints']),
    'sprints': (['project_id'],['projects']),
    'releases': (['project_id'],['projects']),
    'release_issues': (['release_id','issue_id'],['releases','issues']),
    'issue_status_history': (['issue_id'],['issues']),
}

table_structure = []

for child, value in fk.items():
    
    for foreignKey,parentTable in zip(value[0],value[1]):

        prim_key = primary_keys[parentTable]
        
        row = {
            'CHILD TABLE': child,
            'FOREIGN KEY': foreignKey,
            'PARENT TABLE': parentTable,
            'PRIMARY KEY': prim_key[0]
        }
    
        table_structure.append(row)

table_structure_df = pd.DataFrame(table_structure)

table_structure_df

,CHILD TABLE,FOREIGN KEY,PARENT TABLE,PRIMARY KEY
0,projects,owner_team_id,teams,team_id
1,team_members,team_id,teams,team_id
2,epics,project_id,projects,project_id
3,epics,team_id,teams,team_id
4,issues,project_id,projects,project_id
5,issues,team_id,teams,team_id
6,issues,epic_id,epics,epic_id
7,issues,assignee_id,team_members,member_id
8,issues,sprint_id,sprints,sprint_id
9,sprints,project_id,projects,project_id


In [122]:
# identify the primary key datatypes.
pk_datatypes = []

for index, row in table_structure_df.iterrows():
    pk_datatypes.append(
        tables[row['PARENT TABLE']][row['PRIMARY KEY']].dtype
    )

table_structure_df['PK DTYPE'] = pk_datatypes

table_structure_df

,CHILD TABLE,FOREIGN KEY,PARENT TABLE,PRIMARY KEY,PK DTYPE
0,projects,owner_team_id,teams,team_id,int64
1,team_members,team_id,teams,team_id,int64
2,epics,project_id,projects,project_id,int64
3,epics,team_id,teams,team_id,int64
4,issues,project_id,projects,project_id,int64
5,issues,team_id,teams,team_id,int64
6,issues,epic_id,epics,epic_id,int64
7,issues,assignee_id,team_members,member_id,int64
8,issues,sprint_id,sprints,sprint_id,int64
9,sprints,project_id,projects,project_id,int64


In [123]:
# identify the foreign key datatypes
fk_datatypes = []

for index, row in table_structure_df.iterrows():
    fk_datatypes.append(
        tables[row['CHILD TABLE']][row['FOREIGN KEY']].dtype
    )

table_structure_df['FK DTYPE'] = fk_datatypes

table_structure_df


,CHILD TABLE,FOREIGN KEY,PARENT TABLE,PRIMARY KEY,PK DTYPE,FK DTYPE
0,projects,owner_team_id,teams,team_id,int64,int64
1,team_members,team_id,teams,team_id,int64,int64
2,epics,project_id,projects,project_id,int64,int64
3,epics,team_id,teams,team_id,int64,int64
4,issues,project_id,projects,project_id,int64,int64
5,issues,team_id,teams,team_id,int64,int64
6,issues,epic_id,epics,epic_id,int64,int64
7,issues,assignee_id,team_members,member_id,int64,int64
8,issues,sprint_id,sprints,sprint_id,int64,int64
9,sprints,project_id,projects,project_id,int64,int64


In [124]:
#datatype match
table_structure_df['DTYPE MATCH'] = table_structure_df.apply(
    lambda x: "MATCH" 
        if x['PK DTYPE']==x['FK DTYPE'] 
        else 'MISMATCH', 
    axis=1
)


In [125]:
## Find orphan fk values and report count

#loop through for each imported csv and provide count of 
#fk values that are not in the PK values
for index, row in table_structure_df.iterrows():
    table_structure_df.loc[index,'FK ORPHAN COUNT'] = (
            ~(tables[row['CHILD TABLE']][row['FOREIGN KEY']]
            .isin(tables[row['PARENT TABLE']][row['PRIMARY KEY']]))
    ).sum()


table_structure_df

,CHILD TABLE,FOREIGN KEY,PARENT TABLE,PRIMARY KEY,PK DTYPE,FK DTYPE,DTYPE MATCH,FK ORPHAN COUNT
0,projects,owner_team_id,teams,team_id,int64,int64,MATCH,0.0
1,team_members,team_id,teams,team_id,int64,int64,MATCH,0.0
2,epics,project_id,projects,project_id,int64,int64,MATCH,0.0
3,epics,team_id,teams,team_id,int64,int64,MATCH,0.0
4,issues,project_id,projects,project_id,int64,int64,MATCH,0.0
5,issues,team_id,teams,team_id,int64,int64,MATCH,0.0
6,issues,epic_id,epics,epic_id,int64,int64,MATCH,0.0
7,issues,assignee_id,team_members,member_id,int64,int64,MATCH,0.0
8,issues,sprint_id,sprints,sprint_id,int64,int64,MATCH,0.0
9,sprints,project_id,projects,project_id,int64,int64,MATCH,0.0



Foreign key relationships were validated to ensure referential integrity prior to loading the data into PostgreSQL.

Validation included:

- **Datatype consistency:** Confirmed that each foreign key datatype matches the datatype of its referenced primary key.
- **Referential integrity:** Verified that all foreign key values exist in their corresponding parent table primary key.
- **Orphan records:** No orphaned foreign key values were identified.

**Validation Finding:** All defined foreign key relationships passed datatype and referential integrity validation. No remediation is required prior to database load.

### Date Validation

Validate all date/time fields are approrpriate

In [131]:
for name, df in tables.items():
    print('****',name,'****') 
    print(df.dtypes)

**** projects ****
project_id       int64
project_key        str
project_name       str
owner_team_id    int64
dtype: object
**** teams ****
team_id           int64
team_name           str
specialization      str
team_type           str
dtype: object
**** team_members ****
member_id    int64
team_id      int64
role           str
seniority      str
dtype: object
**** epics ****
epic_id       int64
project_id    int64
team_id       int64
epic_key        str
epic_name       str
dtype: object
**** issues ****
issue_id          int64
issue_key           str
project_id        int64
team_id           int64
epic_id           int64
assignee_id       int64
issue_type          str
priority            str
summary             str
created_at          str
resolved_at         str
story_points      int64
current_status      str
sprint_id         int64
dtype: object
**** issue_status_history ****
issue_id      int64
status          str
changed_at      str
dtype: object
**** sprints ****
sprint_id      i

In [135]:
date_columns = {
    'issues': ['created_at','resolved_at'],
    'issue_status_history': ['changed_at'],
    'sprints': ['start_date','end_date'],
    'releases': ['release_date']
}

nat_count = 0

for name, date_cols in date_columns.items():
    print('****',name,'****')
    
    for date_col in date_cols:
        converted = pd.to_datetime(
            tables[name][date_col],
            errors='coerce'
        )

        nat_count += converted.isna().sum()
        #print(date_col, converted)
print(nat_count)

**** issues ****
**** issue_status_history ****
**** sprints ****
**** releases ****
85


Date Validation is complete.  I've identified all possible date/time columns and confirmed they were imported as strings.  They have been identified and are convertable to date/time.  NaT count is 85 which reconciles to the 85 null values previously reported.

## Export Processed Data

Validated and cleaned datasets are exported to the processed data directory for loading into PostgreSQL.

In [140]:
## set export file path and export each dataframe as new processed file
export_path = '../../data/processed/'

for name, df in tables.items():
    print('Exporting ' + export_path + name + '.csv .......')
    
    try:
        df.to_csv(export_path + name + '.csv', index=False)
        print('Successfully Exported ' + export_path + name + '.csv .......')

    except Exception as e:
        print('Error Exporting ' + export_path + name + '.csv;  Aborting process.')
        print(e)
        break

Exporting ../../data/processed/projects.csv .......
Successfully Exported ../../data/processed/projects.csv .......
Exporting ../../data/processed/teams.csv .......
Successfully Exported ../../data/processed/teams.csv .......
Exporting ../../data/processed/team_members.csv .......
Successfully Exported ../../data/processed/team_members.csv .......
Exporting ../../data/processed/epics.csv .......
Successfully Exported ../../data/processed/epics.csv .......
Exporting ../../data/processed/issues.csv .......
Successfully Exported ../../data/processed/issues.csv .......
Exporting ../../data/processed/issue_status_history.csv .......
Successfully Exported ../../data/processed/issue_status_history.csv .......
Exporting ../../data/processed/sprints.csv .......
Successfully Exported ../../data/processed/sprints.csv .......
Exporting ../../data/processed/releases.csv .......
Successfully Exported ../../data/processed/releases.csv .......
Exporting ../../data/processed/release_issues.csv .......


### Load Processed csv for table import

Process logic:
- Load processed data from csv
- execute sql load scripts
- validate data

In [141]:
## set data path
data_path = '../../data/processed/'

## load each csv into a dataframe
processed_projects = pd.read_csv(data_path + 'projects.csv')
processed_teams = pd.read_csv(data_path + 'teams.csv')
processed_team_members = pd.read_csv(data_path + 'team_members.csv')
processed_epics = pd.read_csv(data_path + 'epics.csv')
processed_issues = pd.read_csv(data_path + 'issues.csv')
processed_issue_status_history = pd.read_csv(data_path + 'issue_status_history.csv')
processed_sprints = pd.read_csv(data_path + 'sprints.csv')
processed_releases = pd.read_csv(data_path + 'releases.csv')
processed_release_issues = pd.read_csv(data_path + 'release_issues.csv')

In [159]:
## store processed tables in dictionary
processed_tables = {
    'projects': processed_projects,
    'teams': processed_teams,
    'team_members': processed_team_members,
    'epics': processed_epics,
    'issues': processed_issues,
    'issue_status_history': processed_issue_status_history,
    'sprints': processed_sprints,
    'releases': processed_releases,
    'release_issues': processed_release_issues
}

for name, df in processed_tables.items():
    print('*******************', name, '*******************')
    print(df.shape)
    print('**** Columns in Table ****')
    print(df.columns.tolist())
    print('**** Columns: Data Type ****')
    print(df.dtypes)

******************* projects *******************
(2, 4)
**** Columns in Table ****
['project_id', 'project_key', 'project_name', 'owner_team_id']
**** Columns: Data Type ****
project_id       int64
project_key        str
project_name       str
owner_team_id    int64
dtype: object
******************* teams *******************
(4, 4)
**** Columns in Table ****
['team_id', 'team_name', 'specialization', 'team_type']
**** Columns: Data Type ****
team_id           int64
team_name           str
specialization      str
team_type           str
dtype: object
******************* team_members *******************
(18, 4)
**** Columns in Table ****
['member_id', 'team_id', 'role', 'seniority']
**** Columns: Data Type ****
member_id    int64
team_id      int64
role           str
seniority      str
dtype: object
******************* epics *******************
(50, 5)
**** Columns in Table ****
['epic_id', 'project_id', 'team_id', 'epic_key', 'epic_name']
**** Columns: Data Type ****
epic_id       int64

In [166]:
print(issues['priority'].unique())

<StringArray>
['P3', 'P2', 'P1', 'P4']
Length: 4, dtype: str
